# LA Studio TTS — Kokoro Vietnamese

This notebook loads exactly `kokoro-vietnamese` (`contextboxai/Kokoro-Vietnamese`) on CUDA.
It does not use API Gateway and refuses every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's TTS panel.


In [ ]:
!nvidia-smi
%pip install -q "git+https://github.com/iamdinhthuan/Kokoro-Vietnamese.git@a249afe5555a" "onnxruntime-gpu==1.22.0" "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_tts_worker.py')
WORKER.write_text('import io\nimport os\nimport threading\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_TTS_TOKEN"]\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\n\nclass SpeechRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    voice: str = Field(default="auto", max_length=160)\n    language: str = Field(default="auto", max_length=40)\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\n    response_format: str = "wav"\n    settings: dict = Field(default_factory=dict)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef wav_response(samples, sample_rate: int):\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\n    if audio.size == 0:\n        raise RuntimeError("the selected model returned no audio")\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise HTTPException(status_code=413, detail="generated audio exceeds the five minute output limit")\n    if not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned non-finite audio")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    output = io.BytesIO()\n    sf.write(output, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n    return Response(output.getvalue(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\nfrom functools import lru_cache\nfrom kokoro_vietnamese.core import list_voices\nfrom kokoro_vietnamese.onnx_cli import KokoroVietnameseONNX\n\nMODEL_ID = "kokoro-vietnamese"\nMODEL_NAME = "Kokoro Vietnamese"\nUPSTREAM_MODEL = "contextboxai/Kokoro-Vietnamese"\nSUPPORTED_LANGUAGES = ["vi"]\nSUPPORTED_VOICES = list(list_voices()) or ["diem_trinh"]\n\n@lru_cache(maxsize=24)\ndef model_for(voice: str):\n    runtime = KokoroVietnameseONNX(voice=voice, device="cuda")\n    if "CUDAExecutionProvider" not in runtime.session.get_providers():\n        raise RuntimeError("Kokoro Vietnamese ONNX did not activate CUDAExecutionProvider")\n    return runtime\n\nmodel_for(SUPPORTED_VOICES[0])\n\ndef synthesize_exact_model(request: SpeechRequest):\n    if request.language.strip().lower() not in ("", "auto", "vi", "vi-vn"):\n        raise HTTPException(status_code=422, detail="Kokoro Vietnamese supports Vietnamese only")\n    voice = request.voice.strip() or SUPPORTED_VOICES[0]\n    if voice not in SUPPORTED_VOICES:\n        raise HTTPException(status_code=422, detail="unsupported Kokoro Vietnamese voice")\n    audio, _phonemes = model_for(voice).synthesize(request.input, speed=request.speed)\n    return audio, 24000\n\napp = FastAPI(title=f"LA Studio TTS — {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "tts",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "voices": SUPPORTED_VOICES,\n                "formats": ["wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/speech")\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if request.model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{request.model}\'. Open the notebook for the selected model.",\n        )\n    if request.response_format.strip().lower() != "wav":\n        raise HTTPException(status_code=422, detail="this worker returns WAV audio only")\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab TTS worker is busy; retry shortly")\n    try:\n        samples, sample_rate = synthesize_exact_model(request)\n        return wav_response(samples, sample_rate)\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}",\n        ) from error\n    finally:\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
# LA Studio worker launch contract: launch-2026-08-06.1
import json
import os
import queue
import re
import secrets
import signal
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'TTS'
MODEL_ID = 'kokoro-vietnamese'
PORT = 3921
TOKEN_ENV = 'LA_STUDIO_COLAB_TTS_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_TTS_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_TTS_MODEL'
WORKER_LOG = Path('/content/la_studio_tts_worker.log')
WORKER_MODULE = 'la_studio_tts_worker'
WORKER_PYTHON = sys.executable
WORKER_PYTHON_ISOLATED = False
WORKER_ENVIRONMENT = {}
REQUIRES_CUDA = True
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def process_cmdline(pid: int) -> str:
    """Read a Linux process command line without depending on psutil."""
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(
            "utf-8", errors="replace"
        ).strip()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return ""


def all_processes():
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        pid = int(entry.name)
        command = process_cmdline(pid)
        if command:
            yield pid, command


def listening_processes(port: int) -> dict[int, str]:
    """Return PIDs listening on a local TCP port via /proc socket ownership."""
    target_port = f"{port:04X}"
    socket_inodes = set()
    for table_name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(table_name).read_text(encoding="utf-8").splitlines()[1:]
        except FileNotFoundError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) < 10:
                continue
            local_address, state, inode = fields[1], fields[3], fields[9]
            if state == "0A" and local_address.rsplit(":", 1)[-1].upper() == target_port:
                socket_inodes.add(inode)
    if not socket_inodes:
        return {}

    listeners = {}
    for entry in Path("/proc").iterdir():
        if not entry.name.isdigit():
            continue
        try:
            descriptors = (entry / "fd").iterdir()
        except (FileNotFoundError, PermissionError):
            continue
        for descriptor in descriptors:
            try:
                target = os.readlink(descriptor)
            except (FileNotFoundError, PermissionError, OSError):
                continue
            match = re.fullmatch(r"socket:\\[(\\d+)\\]", target)
            if match and match.group(1) in socket_inodes:
                pid = int(entry.name)
                listeners[pid] = process_cmdline(pid)
                break
    return listeners


def stop_pid(pid: int) -> None:
    if pid == os.getpid():
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass


def reclaim_previous_la_studio_worker() -> None:
    """Stop only an older LA Studio worker/tunnel for this exact local port.

    Re-running a Colab cell keeps child processes alive.  The previous launch
    created a new token but aborted before it could replace the old worker,
    forcing users to destroy the whole GPU runtime.  We identify ownership by
    the exact generated module name and never terminate a foreign listener.
    """
    stopped = []
    for pid, command in listening_processes(PORT).items():
        if WORKER_MODULE in command and "uvicorn" in command:
            stop_pid(pid)
            stopped.append(f"worker PID {pid}")

    endpoint = f"http://127.0.0.1:{PORT}"
    for pid, command in all_processes():
        if ("cloudflared" in command and "tunnel" in command and endpoint in command):
            stop_pid(pid)
            stopped.append(f"tunnel PID {pid}")

    deadline = time.monotonic() + 12
    while port_is_occupied(PORT) and time.monotonic() < deadline:
        time.sleep(0.2)
    if stopped:
        print("Stopped previous LA Studio " + ", ".join(stopped) + ".")

    if port_is_occupied(PORT):
        listeners = listening_processes(PORT)
        foreign_pids = sorted(listeners) or ["unknown"]
        raise RuntimeError(
            f"Port {PORT} is occupied by a process that is not the previous LA Studio "
            f"{CAPABILITY_LABEL} worker (PID(s): {', '.join(map(str, foreign_pids))}). "
            "Choose a fresh Colab runtime rather than terminating an unrelated process."
        )


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


reclaim_previous_la_studio_worker()

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
env.update(WORKER_ENVIRONMENT)
if WORKER_PYTHON_ISOLATED:
    # Do not let Colab's global site-packages or a notebook-level PYTHONPATH
    # bleed into a dedicated worker virtual environment.
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [WORKER_PYTHON, "-m", "uvicorn", 'la_studio_tts_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    worker_kind = "exact CUDA" if REQUIRES_CUDA else "dedicated Colab CPU"
    print(f"Starting {worker_kind} {CAPABILITY_LABEL} worker.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower()
                        == ("cuda" if REQUIRES_CUDA else "colab-cpu")
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print(worker_kind.title() + " worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print(f"Waiting for the {worker_kind} worker...", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The {worker_kind} {CAPABILITY_LABEL} worker did not become ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
